# Inspectioning the desnmaps files

In [ ]:
# Validate all densmap_o*.fits in a catalog output folder

from pathlib import Path
import re
import numpy as np
from astropy.io import fits

base_path = Path("/home/luigi/linea/hipscatalog_gen/outputs/test")

pat = re.compile(r"densmap_o(\d+)\.fits$")
files = sorted(
    [p for p in base_path.glob("densmap_o*.fits") if pat.match(p.name)],
    key=lambda p: int(pat.match(p.name).group(1)),
)

if not files:
    raise FileNotFoundError(f"No densmap_o*.fits found in {base_path}")

rows_sum = []
issues = []

print(f"Found {len(files)} densmap files in: {base_path}\n")

for p in files:
    order = int(pat.match(p.name).group(1))
    expected_npix = 12 * (4 ** order)  # HEALPix npix for given order
    
    with fits.open(p, memmap=True) as hdul:
        if len(hdul) < 2:
            issues.append((p.name, "Missing extension HDU[1]"))
            continue
        if "VALUE" not in hdul[1].columns.names:
            issues.append((p.name, "Missing VALUE column"))
            continue

        values = np.asarray(hdul[1].data["VALUE"])
        n = values.size
        s = int(values.astype(np.int64).sum())
        nnz = int(np.count_nonzero(values))

        ok_npix = (n == expected_npix)
        if not ok_npix:
            issues.append((p.name, f"npix mismatch: got {n}, expected {expected_npix}"))

        rows_sum.append((order, s))
        print(
            f"order={order:2d} | npix={n:,} {'OK' if ok_npix else 'BAD'} "
            f"| nnz={nnz:,} | sum={s:,}"
        )

print("\nChecks:")
# Sum of counts should be constant across orders (same number of selected rows)
unique_sums = sorted({s for _, s in rows_sum})
print(f"- Constant total-count sum across orders: {'OK' if len(unique_sums)==1 else 'BAD'}")
if len(unique_sums) != 1:
    print("  sums by order:", rows_sum)

# Consecutive orders check
orders = [o for o, _ in rows_sum]
is_consecutive = (orders == list(range(min(orders), max(orders)+1)))
print(f"- Orders are consecutive: {'OK' if is_consecutive else 'BAD'}")
if not is_consecutive:
    print("  found orders:", orders)

if issues:
    print("\nIssues found:")
    for name, msg in issues:
        print(f"- {name}: {msg}")
else:
    print("\nNo structural issues found in densmaps.")

# Inspectioning the MOC files

In [ ]:
# Quick validation: Moc.fits vs Moc.json vs densmap_o<moc_order>.fits
# Includes path checks with friendly errors.

from pathlib import Path
import json
import numpy as np
from astropy.io import fits
from mocpy import MOC

base_path = Path("/home/luigi/linea/hipscatalog_gen/outputs/test")  # adjust if needed
moc_order = 11                                                        # adjust here

moc_fits_path = base_path / "Moc.fits"
moc_json_path = base_path / "Moc.json"
densmap_path = base_path / f"densmap_o{moc_order}.fits"

# Path checks
for p in [moc_fits_path, moc_json_path, densmap_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"File not found: {p}\n"
            f"Current notebook cwd: {Path.cwd()}\n"
            "Tip: use an absolute base_path or check your notebook working directory."
        )

# 1) Read MOC from FITS and JSON
moc_fits = MOC.from_fits(str(moc_fits_path))
with moc_json_path.open("r", encoding="utf-8") as f:
    moc_json_obj = json.load(f)
moc_json = MOC.from_json(moc_json_obj)

# 2) Compare MOC FITS vs JSON
fits_cells = moc_fits.flatten()
json_cells = moc_json.flatten()

same_cells = (len(fits_cells) == len(json_cells)) and np.array_equal(fits_cells, json_cells)
same_sky_fraction = np.isclose(float(moc_fits.sky_fraction), float(moc_json.sky_fraction))

# 3) Compare with densmap non-zero cells at moc_order
with fits.open(densmap_path, memmap=True) as hdul:
    counts = np.asarray(hdul[1].data["VALUE"], dtype=np.int64)

dens_nonzero = np.flatnonzero(counts > 0)
same_vs_densmap = (moc_fits.max_order == moc_order) and np.array_equal(fits_cells, dens_nonzero)

# 4) Report
print(f"Base path: {base_path}")
print(f"Requested moc_order: {moc_order}\n")
print(f"MOC FITS max_order: {moc_fits.max_order}")
print(f"MOC JSON max_order: {moc_json.max_order}")
print(f"Densmap non-zero pixels (order {moc_order}): {len(dens_nonzero)}")
print(f"MOC FITS cells (flatten): {len(fits_cells)}")
print(f"MOC JSON cells (flatten): {len(json_cells)}\n")
print(f"[OK] FITS == JSON (cells): {same_cells}")
print(f"[OK] FITS == JSON (sky_fraction): {same_sky_fraction}")
print(f"[OK] MOC FITS matches densmap_o{moc_order}.fits: {same_vs_densmap}")


In [ ]:
# Plot MOC (Moc.fits) in a Mollweide view

from pathlib import Path
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from mocpy import MOC

base_path = Path("/home/luigi/linea/hipscatalog_gen/outputs/test")  # adjust if needed
moc_fits_path = base_path / "Moc.fits"

if not moc_fits_path.exists():
    raise FileNotFoundError(f"File not found: {moc_fits_path}")

moc = MOC.from_fits(str(moc_fits_path))

fig = plt.figure(figsize=(10, 6))
wcs = WCS(
    {
        "NAXIS": 2,
        "NAXIS1": 1200,
        "NAXIS2": 600,
        "CTYPE1": "RA---MOL",
        "CTYPE2": "DEC--MOL",
        "CUNIT1": "deg",
        "CUNIT2": "deg",
        "CRVAL1": 0.0,
        "CRVAL2": 0.0,
        "CRPIX1": 600.0,
        "CRPIX2": 300.0,
        "CDELT1": -0.3,
        "CDELT2": 0.3,
    }
)

ax = fig.add_subplot(111, projection=wcs)
ax.grid(color="lightgray", linestyle="--", linewidth=0.5)

moc.fill(ax=ax, wcs=wcs, alpha=0.55, color="tab:blue")
moc.border(ax=ax, wcs=wcs, color="black", linewidth=0.4)

ax.set_title(f"MOC footprint (max_order={moc.max_order})")
ax.set_xlabel("RA")
ax.set_ylabel("Dec")
plt.tight_layout()
plt.show()